In [ ]:
import requests
from bs4 import BeautifulSoup
from supabase import create_client
import os

SUPABSE_URL = os.getenv("SUPABASE_URL")
SUPABSE_KEY = os.getenv("SUPABASE_KEY")
TRAINER_WEB_USERNAME = os.getenv("TRAINER_WEB_USERNAME")
TRAINER_WEB_PASSWORD = os.getenv("TRAINER_WEB_PASSWORD")

DIVISIONS = ["master", "senior", "junior"]

supabase = create_client(SUPABSE_URL, SUPABSE_KEY)
session = requests.Session()

headers = {
  'accept': 'text/html,application/xhtml+xml,application/xml;q=0.9,image/avif,image/webp,image/apng,*/*;q=0.8',
  'content-type': 'application/x-www-form-urlencoded',
  'origin': 'https://asia.pokemon-card.com',
  'referer': 'https://asia.pokemon-card.com/ph/login/',
  'user-agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/145.0.0.0 Safari/537.36'
}

login_payload = {
  "redirectTo": "",
  "email": TRAINER_WEB_USERNAME,
  "password": TRAINER_WEB_PASSWORD
}

login_res = session.post("https://asia.pokemon-card.com/ph/login/", headers=headers, data=login_payload)

if login_res.status_code == 200:
  print("✅ Login successful (or session established)")
  
  trainers_data = []

  for division in DIVISIONS:
    ranking_url = f"https://asia.pokemon-card.com/ph/mypage/ranking/{division}/"
    res = session.get(ranking_url, headers=headers)
    soup = BeautifulSoup(res.text, 'html.parser')
    
    for row in soup.select(".userInformation"):
      text = row.get_text(separator=" ", strip=True)
      data = text.split(" ")
      
      if id_match:
        trainer_id = data.pop()
        nickname = data.pop()
        
        trainers_data.append({
          "id": trainer_id,
          "nickname": nickname,
          "division": division,
          "full_name": None,
          "alphabet_name": None
        })

  if trainers_data:
    try:
      result = supabase.table("trainers").upsert(trainers_data).execute()
      print(f"🚀 Successfully synced {len(trainers_data)} trainers to Supabase.")
    except Exception as e:
      print(f"❌ Supabase Error: {e}")
  else:
    print("⚠️ No trainer data found. Check if selectors need updating.")
else:
  print(f"❌ Login failed with status: {login_res.status_code}")



✅ Login successful (or session established)
🚀 Successfully synced 1522 trainers to Supabase.
